Step 1 — Convert labeled image → candidate planting points
Assumptions

You already have a labeled raster (label_map)

Resolution = gsd meters per pixel

Drone plants one seedpod per point

You may prefer good > fair, ignore poor

In [ ]:
import numpy as np

# Example ML output (H x W)
# 0 = poor, 1 = fair, 2 = good
label_map = np.load("microsite_labels.npy")

gsd = 0.05  # meters per pixel (example: 5 cm)

# Assign planting weights (preference)
WEIGHTS = {
    2: 1.0,   # good
    1: 2.0    # fair (higher cost = less preferred)
}

planting_points = []

H, W = label_map.shape

for i in range(H):
    for j in range(W):
        label = label_map[i, j]
        if label in WEIGHTS:
            x = j * gsd
            y = i * gsd
            planting_points.append((x, y, WEIGHTS[label]))

planting_points = np.array(planting_points)



Now you have:
[x, y, cost_weight]


3. Step 2 — Optional spatial thinning (VERY important)

You must not plant at every pixel.

Use spatial thinning / clustering:

In [ ]:
from sklearn.cluster import KMeans

n_points = 200  # depends on payload & mission
coords = planting_points[:, :2]
weights = planting_points[:, 2]

kmeans = KMeans(n_clusters=n_points, random_state=42)
kmeans.fit(coords)

planting_sites = kmeans.cluster_centers_


Step 3 — Build the OR flight-path model
What kind of OR model is appropriate?

For drones, this is NOT plain TSP.

Best choice (academically defensible):

TSP with turn penalties

Dubins path constraints (minimum turning radius)

We start simple and extensible.

Step 4 — Distance + turn-aware cost function

Distance matrix

In [ ]:
from scipy.spatial.distance import cdist

D = cdist(planting_sites, planting_sites, metric="euclidean")


Turn penalty function

In [ ]:
import math

def turn_penalty(p_prev, p_curr, p_next, min_turn_radius=2.0):
    v1 = p_curr - p_prev
    v2 = p_next - p_curr

    angle = math.acos(
        np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    )

    # Penalize sharp turns
    if angle > math.pi / 2:
        return 10.0
    return 0.0


Step 5 — Solve the path (OR example)

Simple heuristic solver (good for prototyping)

In [ ]:
from python_tsp.heuristics import solve_tsp_local_search

path, distance = solve_tsp_local_search(D)


Step 6 — Evaluate performance (simulation layer)

This is where your 10% improvement claim comes from.

In [ ]:
def path_length(path, points):
    total = 0
    for i in range(len(path) - 1):
        total += np.linalg.norm(
            points[path[i]] - points[path[i + 1]]
        )
    return total

optimized_length = path_length(path, planting_sites)
baseline_length = path_length(np.random.permutation(len(path)), planting_sites)

improvement = (baseline_length - optimized_length) / baseline_length
print(f"Improvement: {improvement * 100:.2f}%")


Next Steps:

Upgrade this to Dubins paths (very strong for drones)

Add altitude, payload, battery constraints

Turn this into a methods section for a paper

Integrate with real GPS / UTM coordinates